# 03 Data Preparation And Deep Learning

This notebook prepares labels and patches, trains the segmentation model, and writes checkpoints to Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys

REPO_PATH = "/content/WaterDetectionProjectRepo"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/parthpatel182003/Sentinel-2.git {REPO_PATH}

sys.path.append(REPO_PATH)

%cd {REPO_PATH}

/content/WaterDetectionProjectRepo


## To Implement

- load labels and imagery
- generate training patches
- split train, validation, and test sets
- define U-Net baseline
- save checkpoints every epoch
- run inference on held-out scenes

In [3]:
import numpy as np
import cv2
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [4]:
# ======================================
# 03 Data Preparation And Deep Learning
# ======================================

"""
Steps:
1. Convert TIFF + masks → patches
2. Create dataset
3. Train UNet model
4. Save best model
5. Visualize predictions
"""

'\nSteps:\n1. Convert TIFF + masks → patches\n2. Create dataset\n3. Train UNet model\n4. Save best model\n5. Visualize predictions\n'

In [5]:
BASE_PATH = "/content/drive/MyDrive/WaterDetectionProject"

image_dir = BASE_PATH + "_exports"
mask_dir = BASE_PATH + "/masks"

patch_img_dir = BASE_PATH + "/dataset/images"
patch_mask_dir = BASE_PATH + "/dataset/masks"

os.makedirs(patch_img_dir, exist_ok=True)
os.makedirs(patch_mask_dir, exist_ok=True)

print("Paths ready ✅")

Paths ready ✅


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

sample_mask = os.listdir(mask_dir)[5]  # pick any new scene
mask_path = os.path.join(mask_dir, sample_mask)

mask = cv2.imread(mask_path, 0)

print("Mask sum:", np.sum(mask))

plt.imshow(mask, cmap="gray")
plt.title(sample_mask)
plt.show()


Mask sum: 257306475


In [ ]:
import rasterio

def create_patches(image_path, mask_path, patch_size=128, stride=64):
    try:
        with rasterio.open(image_path) as src:
            blue = src.read(1)
            green = src.read(2)
            red = src.read(3)

        img = np.stack([red, green, blue], axis=-1)

        mask = cv2.imread(mask_path, 0)

        if mask is None:
            print("Mask load failed:", mask_path)
            return 0

    except:
        print("Skipping (read error):", image_path)
        return 0

    h, w, _ = img.shape
    count = 0

    for i in range(0, h, stride):
        for j in range(0, w, stride):

            if i + patch_size > h or j + patch_size > w:
                continue

            patch_img = img[i:i+patch_size, j:j+patch_size]
            patch_mask = mask[i:i+patch_size, j:j+patch_size]

            # 🔥 FINAL FILTER
            water_pixels = np.sum(patch_mask > 0)

            if water_pixels > 200:
                img_name = f"{os.path.basename(image_path).replace('.tif','')}_{i}_{j}_img.png"
                mask_name = img_name.replace("img", "mask")

                # normalize before saving
                patch_img = (patch_img / (np.max(patch_img) + 1e-6) * 255).astype(np.uint8)

                cv2.imwrite(os.path.join(patch_img_dir, img_name), patch_img)
                cv2.imwrite(os.path.join(patch_mask_dir, mask_name), patch_mask)

                count += 1

    return count

In [9]:
total_patches = 0

files = sorted([f for f in os.listdir(image_dir) if f.endswith(".tif")])

for idx, file in enumerate(files):
    img_path = os.path.join(image_dir, file)
    mask_path = os.path.join(mask_dir, file.replace(".tif", "_water_mask_mndwi.tif"))

    print(f"[{idx+1}/{len(files)}] Processing: {file}")

    patches = create_patches(img_path, mask_path)
    total_patches += patches

print("\nTotal patches created:", total_patches)

[1/14] Processing: kuttanad_kerala_scene_1.tif


KeyboardInterrupt: 

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import os
import numpy as np
import random



In [ ]:
class WaterDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.imgs = sorted(os.listdir(img_dir))
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]

        img = cv2.imread(os.path.join(self.img_dir, img_name))
        mask = cv2.imread(os.path.join(self.mask_dir, img_name.replace("img", "mask")), 0)

        # Augmentation
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
            mask = cv2.flip(mask, 1)

        if random.random() > 0.5:
            img = cv2.flip(img, 0)
            mask = cv2.flip(mask, 0)

        img = img / 255.0
        mask = mask / 255.0

        img = torch.tensor(img).permute(2,0,1).float()
        mask = torch.tensor(mask).unsqueeze(0).float()

        return img, mask

In [ ]:
dataset = WaterDataset(patch_img_dir, patch_mask_dir)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Total samples:", len(dataset))

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.pool = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )

        self.up = nn.ConvTranspose2d(64, 32, 2, stride=2)

        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))

        x3 = self.up(x2)
        x4 = torch.cat([x1, x3], dim=1)

        x5 = self.dec1(x4)
        return self.out(x5)  # no sigmoid

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

checkpoint_path = BASE_PATH + "/best_model.pth"

In [ ]:
BASE_PATH = "/content/drive/MyDrive/WaterDetectionProject"

patch_img_dir = BASE_PATH + "/dataset/images"
patch_mask_dir = BASE_PATH + "/dataset/masks"

print("Images:", len(os.listdir(patch_img_dir)))
print("Masks:", len(os.listdir(patch_mask_dir)))

In [ ]:
from tqdm import tqdm

best_loss = float("inf")
patience = 3
counter = 0

epochs = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

    for imgs, masks in loop:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 🔥 show live loss
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f"\nEpoch {epoch+1}/{epochs}, Avg Loss: {avg_loss:.4f}")

    # Checkpoint
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), checkpoint_path)
        print("✅ Model saved")
        counter = 0
    else:
        counter += 1
        print(f"No improvement ({counter}/3)")

    # Early stopping
    if counter >= patience:
        print("🛑 Early stopping")
        break

In [ ]:
model.load_state_dict(torch.load(checkpoint_path))
model.eval()

imgs, masks = next(iter(loader))
imgs = imgs.to(device)

with torch.no_grad():
    preds = model(imgs)

preds = torch.sigmoid(preds)

imgs = imgs.cpu().numpy()
masks = masks.numpy()
preds = preds.cpu().numpy()

In [ ]:
for i in range(3):
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(imgs[i].transpose(1,2,0))
    plt.title("Input")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(masks[i][0], cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(preds[i][0] > 0.5, cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

    plt.show()

In [ ]:
def compute_iou(pred, mask):
    pred = pred > 0.5
    mask = mask > 0.5

    intersection = (pred & mask).sum()
    union = (pred | mask).sum()

    return intersection / (union + 1e-6)

ious = []

for i in range(len(preds)):
    ious.append(compute_iou(preds[i][0], masks[i][0]))

print("Average IoU:", np.mean(ious))